# PG-Stack · Step 4 — Patient-Graph Meta-Learner (the main contribution)

**Idea:** one graph per patient. **Node = lesion.** The GNN sees every lesion of the patient at once and learns the *ugly-duckling* comparison instead of using hand-made patient-normalised features.

**Node features** (per lesion)
- logits of the 3 lesion-only base learners (OOF from Steps 1–2: tab_lesion, emb_lr, cnn)
- raw lesion features (~70 colour / shape / border / size numbers, one-hot site & sex)
- DINOv2 image embedding (PCA-32)

**Edges** (within a patient only)
- *appearance* edges: k nearest neighbours in DINOv2 embedding space
- *anatomic* edges: k nearest neighbours in 3D body coordinates (x, y, z)

**Model:** input MLP → 2 relational GATv2 layers (one conv per edge type) → patient context `[h, h − mean_patient(h), max_patient(h) − h]` → MLP → malignancy logit.

**Variants (ablations run in this notebook)**
| Variant | Graph edges | Patient context pooling | Purpose |
|---|---|---|---|
| `node_mlp` | ✗ | ✗ | same node features, no patient info → is the graph needed? |
| `set_ctx` | ✗ | ✓ | DeepSets-style patient context, no edges |
| `pgstack` | ✓ | ✓ | **full PG-Stack** |
| `pgstack_plus` | ✓ | ✓ | + tab_pnorm logit as an extra node feature |

**Protocol:** for outer fold *f*: train on 3 folds, early-stop on fold *(f+1) mod 5* (inner validation, by pAUC80), predict fold *f*. Averaged over seeds. Same patient folds as all earlier steps → no leakage.

**Runs in the same session:** reads from / writes to `/kaggle/working`. **GPU + Internet ON** (installs PyTorch Geometric). Runtime ≈ 60–90 min with 1 seed. Results are saved after every variant.

In [ ]:
import subprocess, sys
try:
    import torch_geometric
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"], check=True)
    import torch_geometric
print("PyG", torch_geometric.__version__)

In [ ]:
import os, time, gc, copy, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GraphLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
sys.path.insert(0, WORK)
from pgstack_eval import *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [0]                       # use [0, 1, 2] for the final paper run
VARIANTS = ["pgstack", "node_mlp", "set_ctx", "pgstack_plus"]   # main model first
K_APP, K_POS = 10, 10             # neighbours per edge type
HIDDEN, HEADS, DROPOUT = 64, 4, 0.2
LR, WD = 1e-3, 1e-4
MAX_EPOCHS, PATIENCE = 40, 8
BATCH_PATIENTS = 16
POS_WEIGHT = 20.0

for f in ["oof_step3.csv", "lesion_features.parquet", "emb_pca32.npy", "pgstack_eval.py"]:
    assert os.path.exists(f"{WORK}/{f}"), f"{f} missing - run Steps 1-3 first in this session"
print("device:", DEVICE)

## 1. Load and align everything (row order = Step-1 order)

In [ ]:
oof  = pd.read_csv(f"{WORK}/oof_step3.csv")
feat = pd.read_parquet(f"{WORK}/lesion_features.parquet")
emb  = np.load(f"{WORK}/emb_pca32.npy").astype(np.float32)
assert (oof.isic_id.values == feat.isic_id.values).all() and len(emb) == len(oof), "row order mismatch"

y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
N_FOLDS = len(np.unique(folds))

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

BASE = ["oof_tab_lesion", "oof_emb_lr", "oof_cnn"]
X_pred = np.stack([logit(oof[c].values) for c in BASE], 1).astype(np.float32)
X_pnorm = logit(oof["oof_tab_pnorm"].values)[:, None].astype(np.float32)

cat_cols = [c for c in ["anatom_site_general", "tbp_lv_location_simple", "sex"] if c in feat.columns]
num_cols = [c for c in feat.columns if c not in ["isic_id", "patient_id", "fold", "target"] + cat_cols]
X_num = feat[num_cols].apply(pd.to_numeric, errors="coerce")
X_num = X_num.fillna(X_num.median()).values.astype(np.float32)
X_cat = pd.get_dummies(feat[cat_cols].astype(str), dtype=np.float32).values

X_main = np.concatenate([X_pred, X_num, X_cat, emb], 1)          # pgstack / ablations
X_plus = np.concatenate([X_main, X_pnorm], 1)                    # pgstack_plus
POS = feat[["tbp_lv_x", "tbp_lv_y", "tbp_lv_z"]].values.astype(np.float32)
print("node features:", X_main.shape[1], "(preds 3 | numeric", X_num.shape[1], "| cat", X_cat.shape[1], "| emb", emb.shape[1], ")")

## 2. Build one graph per patient (appearance + anatomic kNN edges)

In [ ]:
def knn_edges(Z, k):
    n = len(Z)
    if n < 2: return np.zeros((2, 0), dtype=np.int64)
    kk = min(k, n - 1)
    nn_ = NearestNeighbors(n_neighbors=kk + 1).fit(Z)
    _, idx = nn_.kneighbors(Z)
    src = idx[:, 1:].reshape(-1); dst = np.repeat(np.arange(n), kk)
    return np.stack([src, dst]).astype(np.int64)     # message flows neighbour -> lesion

t0 = time.time()
groups = pd.Series(np.arange(len(oof))).groupby(pid).indices
patient_ids = list(groups.keys())
EDGES, ROWS, PFOLD = {}, {}, {}
for p in patient_ids:
    r = np.sort(groups[p]); ROWS[p] = r; PFOLD[p] = folds[r[0]]
    EDGES[p] = (knn_edges(emb[r], K_APP), knn_edges(POS[r], K_POS))
n_e = sum(a.shape[1] + b.shape[1] for a, b in EDGES.values())
print(f"{len(patient_ids)} patient graphs, {n_e:,} edges, built in {time.time()-t0:.0f}s")

def make_graphs(X, plist):
    out = []
    for p in plist:
        r = ROWS[p]; ea, ep = EDGES[p]
        out.append(Data(x=torch.from_numpy(X[r]), y=torch.from_numpy(y[r].astype(np.float32)),
                        edge_index=torch.from_numpy(ea), pos_edge_index=torch.from_numpy(ep),
                        row=torch.from_numpy(r), num_nodes=len(r)))
    return out

## 3. Model

In [ ]:
class PGNet(nn.Module):
    def __init__(self, d_in, use_graph=True, use_ctx=True, h=HIDDEN, heads=HEADS, drop=DROPOUT):
        super().__init__()
        self.use_graph, self.use_ctx = use_graph, use_ctx
        self.register_buffer("mu", torch.zeros(d_in)); self.register_buffer("sd", torch.ones(d_in))
        self.inp = nn.Sequential(nn.Linear(d_in, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(drop))
        if use_graph:
            self.app = nn.ModuleList([GATv2Conv(h, h // heads, heads=heads, dropout=drop) for _ in range(2)])
            self.pos = nn.ModuleList([GATv2Conv(h, h // heads, heads=heads, dropout=drop) for _ in range(2)])
            self.norm = nn.ModuleList([nn.LayerNorm(h) for _ in range(2)])
        d_head = h * 3 if use_ctx else h
        self.head = nn.Sequential(nn.Linear(d_head, h), nn.GELU(), nn.Dropout(drop), nn.Linear(h, 1))

    def set_norm(self, mu, sd):
        self.mu.copy_(torch.as_tensor(mu)); self.sd.copy_(torch.as_tensor(sd))

    def forward(self, d):
        x = ((d.x - self.mu) / self.sd).clamp(-5, 5)
        h = self.inp(x)
        if self.use_graph:
            for a, p, ln in zip(self.app, self.pos, self.norm):
                h = ln(h + torch.nn.functional.gelu(a(h, d.edge_index) + p(h, d.pos_edge_index)))
        if self.use_ctx:
            m  = global_mean_pool(h, d.batch)[d.batch]
            mx = global_max_pool(h, d.batch)[d.batch]
            h = torch.cat([h, h - m, mx - h], 1)       # "how different is this lesion from its patient?"
        return self.head(h).squeeze(1)

VARIANT_CFG = {"node_mlp": dict(use_graph=False, use_ctx=False, X="main"),
               "set_ctx":  dict(use_graph=False, use_ctx=True,  X="main"),
               "pgstack":  dict(use_graph=True,  use_ctx=True,  X="main"),
               "pgstack_plus": dict(use_graph=True, use_ctx=True, X="plus")}

## 4. Training with inner early stopping

In [ ]:
@torch.no_grad()
def predict(model, graphs):
    model.eval(); rows, preds = [], []
    for b in GraphLoader(graphs, batch_size=BATCH_PATIENTS * 2):
        b = b.to(DEVICE)
        preds.append(torch.sigmoid(model(b)).cpu().numpy()); rows.append(b.row.cpu().numpy())
    return np.concatenate(rows), np.concatenate(preds)

def train_one(X, cfg, f, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    vf = (f + 1) % N_FOLDS
    tr_p = [p for p in patient_ids if PFOLD[p] not in (f, vf)]
    va_p = [p for p in patient_ids if PFOLD[p] == vf]
    te_p = [p for p in patient_ids if PFOLD[p] == f]
    tr_rows = np.concatenate([ROWS[p] for p in tr_p])
    mu, sd = X[tr_rows].mean(0), X[tr_rows].std(0) + 1e-6           # normalisation from training nodes only
    g_tr, g_va, g_te = make_graphs(X, tr_p), make_graphs(X, va_p), make_graphs(X, te_p)

    model = PGNet(X.shape[1], cfg["use_graph"], cfg["use_ctx"]).to(DEVICE)
    model.set_norm(mu, sd)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))
    best, best_state, bad = -1, None, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        for b in GraphLoader(g_tr, batch_size=BATCH_PATIENTS, shuffle=True):
            b = b.to(DEVICE); opt.zero_grad()
            loss = lossf(model(b), b.y); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        r, p = predict(model, g_va)
        s = pauc_80(y[r], p)
        if s > best: best, best_state, bad = s, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    r, p = predict(model, g_te)
    return r, p, best, ep + 1

results_path = f"{WORK}/oof_step4.csv"
out = oof[["isic_id", "patient_id", "fold", "target"]].copy()
if os.path.exists(results_path):                                   # resume if a variant already finished
    prev = pd.read_csv(results_path)
    for c in prev.columns:
        if c.startswith("pg_"): out[c] = prev[c].values

for name in VARIANTS:
    col = f"pg_{name}"
    if col in out.columns:
        print(f"{name}: already done, skipping"); continue
    cfg = VARIANT_CFG[name]; X = X_main if cfg["X"] == "main" else X_plus
    pred = np.zeros(len(oof)); t0 = time.time()
    for f in range(N_FOLDS):
        acc = np.zeros(len(oof)); mask = np.zeros(len(oof), bool)
        for s in SEEDS:
            r, p, vbest, n_ep = train_one(X, cfg, f, s)
            acc[r] += pd.Series(p).rank(pct=True).values / len(SEEDS); mask[r] = True
            print(f"  {name} fold {f} seed {s}: inner-val pAUC={vbest:.4f} ({n_ep} ep)")
        pred[mask] = acc[mask]
        print(f"[{name}] fold {f}: test pAUC={pauc_80(y[mask], pred[mask]):.4f}")
    out[col] = pred
    out.to_csv(results_path, index=False)                          # saved after every variant
    print(f"== {name}: OOF pAUC80 = {pauc_80(y, pred):.4f}  ({(time.time()-t0)/60:.1f} min)\n")
    gc.collect(); torch.cuda.empty_cache()

## 5. Final comparison: PG-Stack vs every baseline

In [ ]:
allp = oof.merge(out[["isic_id"] + [c for c in out.columns if c.startswith("pg_")]], on="isic_id")
MODELS = {"tab_lesion": "oof_tab_lesion", "tab_pnorm": "oof_tab_pnorm", "stack_lr": "stack_lr",
          "stack_xgb_pctx": "stack_xgb_pctx", "stack_xgb_all": "stack_xgb_all"}
MODELS.update({c.replace("pg_", ""): c for c in allp.columns if c.startswith("pg_")})

boot = PatientBootstrap(pid, n_boot=300, seed=42)
rows = []
for name, col in MODELS.items():
    r = full_report(name, y, allp[col].values, pid)
    mu, sd = fold_stats(y, allp[col].values, folds)
    lo, hi = boot.ci(y, allp[col].values)
    r.update(pauc80_fold=f"{mu:.4f} ± {sd:.4f}", pauc80_95CI=f"[{lo:.4f}, {hi:.4f}]")
    rows.append(r)
res = pd.DataFrame(rows).set_index("model")
cols = ["pauc80", "pauc80_95CI", "pauc80_fold", "auroc", "sens_spec95", "spec_sens95", "top5_recall", "top10_recall"]
print(res[cols].round(4).to_string())

In [ ]:
tests = []
pg_models = [m for m in ["pgstack", "pgstack_plus"] if m in MODELS]
for new in pg_models:
    for base in ["tab_pnorm", "stack_lr", "stack_xgb_pctx", "stack_xgb_all"] + (["node_mlp", "set_ctx"] if new == "pgstack" else []):
        if base not in MODELS: continue
        d, (lo, hi), p = boot.paired_test(y, allp[MODELS[new]].values, allp[MODELS[base]].values)
        tests.append(dict(comparison=f"{new} vs {base}", delta_pauc=round(d, 4),
                          ci95=f"[{lo:+.4f}, {hi:+.4f}]", p_value=round(p, 4)))
tests = pd.DataFrame(tests)
print(tests.to_string(index=False))
res[cols].to_csv(f"{WORK}/results_step4.csv"); tests.to_csv(f"{WORK}/tests_step4.csv", index=False)